# 02 – Eksperymenty: modele nadzorowane vs nienadzorowane
Wspólny kod w pakiecie `anomaly/`. Próg wybierany na walidacji, metryki liczone na teście.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from anomaly.data import load_raw, temporal_split
from anomaly.models import REGISTRY
from anomaly.train import train_one
splits = temporal_split(load_raw('../data/raw/creditcard.csv'))

In [ ]:
rows = []
for name in REGISTRY:
    m = train_one(name, splits, seed=42, out_dir='../models')
    rows.append({'model': name, 'kind': m['kind'], **{k: m['metrics'][k] for k in ['pr_auc','roc_auc','precision','recall','f1']},
                 'fit_s': m['fit_seconds'], 'latency_ms': m['predict_latency_ms']})
res = pd.DataFrame(rows).set_index('model').round(4)
res

In [ ]:
# Stabilność: kilka seedów (średnia ± odch. std.)
import numpy as np
seeds = [1, 2, 3, 4, 5]
agg = {}
for name in ['random_forest', 'xgboost', 'isolation_forest']:
    agg[name] = [train_one(name, splits, seed=s, out_dir='../models/_tmp')['metrics']['pr_auc'] for s in seeds]
pd.DataFrame({k: (np.mean(v), np.std(v)) for k, v in agg.items()}, index=['mean PR-AUC', 'std']).T

**TODO (rozdział 5):**
- wariant nienadzorowany *normal-only* (`normal_only=True`) vs całość,
- porównanie `class_weight` vs SMOTE (imbalanced-learn, tylko na train),
- strojenie hiperparametrów (Optuna / RandomizedSearchCV z `average_precision`),
- krzywe ROC i PR do `figures/` (dane w `models/<name>.json` → `curves`),
- analiza kosztu błędów (FN drożej niż FP) i wybór progu,
- opcjonalnie: autoenkoder, drugi zbiór (PaySim / IEEE-CIS).